# Caso de Estudio 3: Optimización de Inventario, Merma y Rentabilidad

## Contexto

Una cadena de cafeterías con cuatro sucursales comercializa productos perecibles de las categorías Panadería, Pastelería, Sándwiches, Ensaladas y Platos preparados.

La empresa ha detectado pérdidas asociadas a sobreproducción, merma, baja rotación de inventario y posibles quiebres de stock. Además, trabaja con distintos proveedores que ofrecen diferentes precios y tiempos de entrega.

El desafío consiste en identificar dónde se están generando las principales pérdidas y encontrar un equilibrio entre disponibilidad de productos, control de inventario y rentabilidad.

## Objetivo del análisis

Analizar el comportamiento de ventas, inventario, merma, rentabilidad y reposiciones para identificar oportunidades de mejora en la gestión de productos perecibles.

El análisis buscará detectar productos con exceso o falta de inventario, estimar posibles pérdidas de demanda por quiebres de stock, evaluar la rentabilidad real después de la merma y comparar proveedores considerando tanto costo como cumplimiento de entrega.

## Preguntas de negocio

### Inventario
1. ¿Qué productos presentan señales de sobreproducción, acumulación de stock y merma recurrente?
2. ¿Qué productos muestran baja rotación de inventario y podrían estar manteniéndose en stock por encima de su demanda real?

### Quiebres de stock
3. ¿Qué productos y sucursales presentan con mayor frecuencia señales de quiebre de stock?
4. ¿En qué casos los posibles quiebres podrían estar asociados a una planificación insuficiente de reposición o a tiempos de entrega demasiado largos?

### Rentabilidad
5. ¿Qué productos generan mayor ganancia después de considerar el costo de las unidades vendidas y las pérdidas por merma?
6. ¿Qué productos combinan alto volumen de ventas y alta ganancia, y cuáles venden mucho pero presentan una rentabilidad poco atractiva?

### Proveedores
7. ¿Qué diferencias existen entre proveedores de un mismo producto en costo unitario y tiempo de entrega?
8. ¿Qué proveedores ofrecen la mejor relación entre costo, rapidez de reposición y cumplimiento de los plazos de entrega?

In [68]:
import pandas as pd
import numpy as np

from collections import defaultdict, deque

rng = np.random.default_rng(42)

In [69]:
fechas = pd.date_range("2026-01-01", "2026-12-31", freq="D")

sucursales = [
    "Ñuñoa",
    "Providencia",
    "Santiago Centro",
    "Maipú"
]

productos_por_categoria = {
    "Panadería": [
        "Croissant mantequilla", "Pan de chocolate", "Marraqueta gourmet",
        "Baguette integral", "Pan de masa madre", "Roll de canela",
        "Brioche", "Focaccia", "Pan de queso",
        "Muffin salado", "Palitos de ajo", "Pan pita"
    ],

    "Pastelería": [
        "Cheesecake frutos rojos", "Torta chocolate", "Tiramisú",
        "Carrot cake", "Brownie", "Pie de limón",
        "Tarta de manzana", "Muffin arándanos", "Galleta NY chocolate",
        "Galleta avena", "Alfajor", "Donut glaseada"
    ],

    "Sándwiches": [
        "Sándwich ave palta", "Sándwich jamón queso", "Sándwich caprese",
        "Sándwich roast beef", "Sándwich veggie", "Wrap pollo",
        "Wrap mediterráneo", "Bagel salmón", "Bagel pavo",
        "Ciabatta italiana", "Club sandwich", "Croissant relleno"
    ],

    "Ensaladas": [
        "Ensalada César", "Ensalada quinoa", "Ensalada mediterránea",
        "Ensalada caprese", "Ensalada pollo", "Ensalada atún",
        "Ensalada veggie", "Ensalada cous cous", "Ensalada pasta",
        "Ensalada garbanzos", "Ensalada palta", "Ensalada griega"
    ],

    "Platos preparados": [
        "Quiche lorraine", "Quiche espinaca", "Lasaña boloñesa",
        "Lasaña vegetariana", "Pasta pesto", "Pasta pomodoro",
        "Bowl pollo", "Bowl veggie", "Sopa zapallo",
        "Sopa tomate", "Pastel de choclo", "Wrap caliente"
    ]
}

vida_util = {
    "Panadería": (1, 3),
    "Pastelería": (2, 5),
    "Sándwiches": (1, 2),
    "Ensaladas": (1, 2),
    "Platos preparados": (1, 3)
}

rango_precios = {
    "Panadería": (1800, 4200),
    "Pastelería": (2800, 6500),
    "Sándwiches": (4200, 7600),
    "Ensaladas": (4500, 8200),
    "Platos preparados": (5200, 9500)
}

rango_demanda = {
    "Panadería": (14, 30),
    "Pastelería": (8, 20),
    "Sándwiches": (10, 24),
    "Ensaladas": (5, 14),
    "Platos preparados": (7, 18)
}

## Construcción de la base de datos

Para el análisis se generarán dos conjuntos de datos relacionados:

- **Operación diaria:** ventas, inventario y merma por producto, sucursal y fecha.
- **Reposiciones:** pedidos realizados a distintos proveedores, incluyendo costos y tiempos de entrega.

La base simulará un año completo de operación para 60 productos distribuidos en cuatro sucursales. Se incorporarán distintos comportamientos operacionales para representar situaciones como sobreproducción, baja demanda, quiebres de stock y diferencias de rentabilidad.

In [70]:
catalogo_productos = []

producto_num = 1

for categoria, lista_productos in productos_por_categoria.items():

    vida_min, vida_max = vida_util[categoria]
    precio_min, precio_max = rango_precios[categoria]
    demanda_min, demanda_max = rango_demanda[categoria]

    for producto in lista_productos:

        precio_venta = int(
            round(rng.uniform(precio_min, precio_max) / 100) * 100
        )

        costo_referencia = int(
            round((precio_venta * rng.uniform(0.42, 0.62)) / 50) * 50
        )

        catalogo_productos.append({
            "producto_id": f"P{producto_num:03d}",
            "producto": producto,
            "categoria": categoria,
            "vida_util_dias": int(rng.integers(vida_min, vida_max + 1)),
            "precio_venta_neto": precio_venta,
            "costo_referencia": costo_referencia,
            "demanda_base_dia": int(
                rng.integers(demanda_min, demanda_max + 1)
            )
        })

        producto_num += 1

catalogo = pd.DataFrame(catalogo_productos)

catalogo.head()

,producto_id,producto,categoria,vida_util_dias,precio_venta_neto,costo_referencia,demanda_base_dia
0,P001,Croissant mantequilla,Panadería,2,3700,1900,28
1,P002,Pan de chocolate,Panadería,2,3500,1550,30
2,P003,Marraqueta gourmet,Panadería,2,3600,2100,16
3,P004,Baguette integral,Panadería,1,2900,1450,29
4,P005,Pan de masa madre,Panadería,2,3300,1950,21


### Patrones operacionales simulados

Para evitar que el conjunto de datos sea completamente aleatorio, se incorporarán distintos comportamientos dentro de la operación.

Algunos productos tendrán mayor riesgo de sobreproducción y merma, otros presentarán baja demanda o mayor riesgo de quiebre de stock. También existirán productos con alto volumen y rentabilidad, junto con otros que venden bien pero presentan márgenes reducidos.

Estos patrones no aparecerán identificados en la base final: deberán ser detectados posteriormente mediante el análisis.

In [71]:
catalogo["factor_demanda"] = 1.00
catalogo["factor_reposicion"] = 1.00
catalogo["factor_merma"] = 1.00
catalogo["perfil_sintetico"] = "Normal"

# Mayor sobreproducción y merma en algunos productos de Pastelería
idx_sobreproduccion = rng.choice(
    catalogo[catalogo["categoria"] == "Pastelería"].index,
    size=6,
    replace=False
)

catalogo.loc[idx_sobreproduccion, "factor_reposicion"] = 1.35
catalogo.loc[idx_sobreproduccion, "factor_merma"] = 1.40
catalogo.loc[idx_sobreproduccion, "perfil_sintetico"] = "Sobreproducción"


# Baja demanda en algunos productos de Ensaladas
idx_baja_demanda = rng.choice(
    catalogo[catalogo["categoria"] == "Ensaladas"].index,
    size=6,
    replace=False
)

catalogo.loc[idx_baja_demanda, "factor_demanda"] = 0.55
catalogo.loc[idx_baja_demanda, "factor_reposicion"] = 1.15
catalogo.loc[idx_baja_demanda, "perfil_sintetico"] = "Baja demanda"


# Alta demanda pero reposición insuficiente en algunos Sándwiches
idx_quiebre = rng.choice(
    catalogo[catalogo["categoria"] == "Sándwiches"].index,
    size=6,
    replace=False
)

catalogo.loc[idx_quiebre, "factor_demanda"] = 1.35
catalogo.loc[idx_quiebre, "factor_reposicion"] = 0.78
catalogo.loc[idx_quiebre, "perfil_sintetico"] = "Riesgo quiebre"


# Seleccionamos productos restantes para crear estrellas
indices_usados = set(idx_sobreproduccion) | set(idx_baja_demanda) | set(idx_quiebre)

indices_disponibles = [
    i for i in catalogo.index
    if i not in indices_usados
]

idx_estrellas = rng.choice(
    indices_disponibles,
    size=6,
    replace=False
)

catalogo.loc[idx_estrellas, "factor_demanda"] = 1.35
catalogo.loc[idx_estrellas, "perfil_sintetico"] = "Estrella"

catalogo.loc[idx_estrellas, "costo_referencia"] = (
    catalogo.loc[idx_estrellas, "precio_venta_neto"]
    * rng.uniform(0.38, 0.48, size=6)
).round(-1).astype(int)


# Productos de alto volumen pero margen reducido
indices_usados = indices_usados | set(idx_estrellas)

indices_disponibles = [
    i for i in catalogo.index
    if i not in indices_usados
]

idx_margen_bajo = rng.choice(
    indices_disponibles,
    size=6,
    replace=False
)

catalogo.loc[idx_margen_bajo, "factor_demanda"] = 1.20
catalogo.loc[idx_margen_bajo, "perfil_sintetico"] = "Margen bajo"

catalogo.loc[idx_margen_bajo, "costo_referencia"] = (
    catalogo.loc[idx_margen_bajo, "precio_venta_neto"]
    * rng.uniform(0.74, 0.82, size=6)
).round(-1).astype(int)

### Proveedores

Cada producto contará con varias alternativas de abastecimiento.

Los proveedores presentarán diferencias en:

- costo unitario,
- tiempo de entrega,
- frecuencia de retrasos.

Esto permitirá analizar si el proveedor con menor precio es necesariamente la alternativa más conveniente cuando también se considera el riesgo de quiebres de stock.

In [72]:
perfil_proveedores = {
    "Proveedor A": {
        "factor_costo": 1.02,
        "dias_entrega": 2,
        "prob_retraso": 0.08
    },

    "Proveedor B": {
        "factor_costo": 0.88,
        "dias_entrega": 5,
        "prob_retraso": 0.32
    },

    "Proveedor C": {
        "factor_costo": 0.95,
        "dias_entrega": 3,
        "prob_retraso": 0.15
    },

    "Proveedor D": {
        "factor_costo": 1.08,
        "dias_entrega": 1,
        "prob_retraso": 0.05
    }
}


filas_proveedores = []

for _, producto in catalogo.iterrows():

    # Todos los productos pueden comprarse a A y B.
    # El tercer proveedor será C o D.
    proveedores_producto = [
        "Proveedor A",
        "Proveedor B",
        rng.choice(["Proveedor C", "Proveedor D"])
    ]

    for proveedor in proveedores_producto:

        perfil = perfil_proveedores[proveedor]

        costo = (
            producto["costo_referencia"]
            * perfil["factor_costo"]
            * rng.uniform(0.97, 1.03)
        )

        filas_proveedores.append({
            "producto_id": producto["producto_id"],
            "producto": producto["producto"],
            "proveedor": proveedor,
            "costo_unitario_proveedor": int(round(costo / 10) * 10),
            "dias_entrega_prometidos": perfil["dias_entrega"]
        })


catalogo_proveedores = pd.DataFrame(filas_proveedores)

catalogo_proveedores.head(9)

,producto_id,producto,proveedor,costo_unitario_proveedor,dias_entrega_prometidos
0,P001,Croissant mantequilla,Proveedor A,1910,2
1,P001,Croissant mantequilla,Proveedor B,1670,5
2,P001,Croissant mantequilla,Proveedor D,2070,1
3,P002,Pan de chocolate,Proveedor A,1420,2
4,P002,Pan de chocolate,Proveedor B,1270,5
5,P002,Pan de chocolate,Proveedor C,1300,3
6,P003,Marraqueta gourmet,Proveedor A,2150,2
7,P003,Marraqueta gourmet,Proveedor B,1860,5
8,P003,Marraqueta gourmet,Proveedor D,2210,1


In [73]:
print("Catálogo de productos:", catalogo.shape)
print("Catálogo de proveedores:", catalogo_proveedores.shape)

display(
    catalogo[
        [
            "producto_id",
            "producto",
            "categoria",
            "vida_util_dias",
            "precio_venta_neto",
            "costo_referencia",
            "demanda_base_dia"
        ]
    ].head()
)

display(catalogo_proveedores.head(9))

Catálogo de productos: (60, 11)
Catálogo de proveedores: (180, 5)


,producto_id,producto,categoria,vida_util_dias,precio_venta_neto,costo_referencia,demanda_base_dia
0,P001,Croissant mantequilla,Panadería,2,3700,1900,28
1,P002,Pan de chocolate,Panadería,2,3500,1410,30
2,P003,Marraqueta gourmet,Panadería,2,3600,2100,16
3,P004,Baguette integral,Panadería,1,2900,1450,29
4,P005,Pan de masa madre,Panadería,2,3300,2470,21


,producto_id,producto,proveedor,costo_unitario_proveedor,dias_entrega_prometidos
0,P001,Croissant mantequilla,Proveedor A,1910,2
1,P001,Croissant mantequilla,Proveedor B,1670,5
2,P001,Croissant mantequilla,Proveedor D,2070,1
3,P002,Pan de chocolate,Proveedor A,1420,2
4,P002,Pan de chocolate,Proveedor B,1270,5
5,P002,Pan de chocolate,Proveedor C,1300,3
6,P003,Marraqueta gourmet,Proveedor A,2150,2
7,P003,Marraqueta gourmet,Proveedor B,1860,5
8,P003,Marraqueta gourmet,Proveedor D,2210,1


## Simulación de la operación diaria

A partir del catálogo de productos y proveedores se simulará la operación de las cuatro sucursales durante un año completo.

La simulación considerará:

- demanda diaria variable por producto y sucursal,
- inventario disponible,
- ventas limitadas por el stock existente,
- vencimiento y merma de productos perecibles,
- decisiones de reposición,
- tiempos de entrega y posibles retrasos de proveedores,
- diferencias de planificación entre productos y sucursales.

Las unidades se consumirán siguiendo una lógica FIFO, priorizando los productos con fecha de vencimiento más próxima.

In [74]:
# Factores de demanda y planificación por sucursal
factor_demanda_sucursal = {
    "Ñuñoa": 1.05,
    "Providencia": 1.10,
    "Santiago Centro": 1.15,
    "Maipú": 0.85
}

factor_reposicion_sucursal = {
    "Ñuñoa": 1.00,
    "Providencia": 1.00,
    "Santiago Centro": 0.95,
    "Maipú": 1.18
}

# Algunos efectos semanales
factor_dia_semana = {
    0: 0.95,   # lunes
    1: 1.00,
    2: 1.00,
    3: 1.03,
    4: 1.12,   # viernes
    5: 1.18,   # sábado
    6: 0.90    # domingo
}

# Tasas base de merma operacional
merma_base_categoria = {
    "Panadería": 0.012,
    "Pastelería": 0.018,
    "Sándwiches": 0.015,
    "Ensaladas": 0.020,
    "Platos preparados": 0.018
}

# Acceso rápido a información de productos
info_productos = catalogo.set_index("producto_id").to_dict("index")

# Proveedores disponibles por producto
proveedores_por_producto = {
    producto_id: grupo.to_dict("records")
    for producto_id, grupo
    in catalogo_proveedores.groupby("producto_id")
}

In [75]:
def quitar_unidades_fifo(inventario, cantidad):
    """
    Retira unidades comenzando por el lote que vence primero.
    Devuelve la cantidad retirada y su costo total.
    """
    cantidad_restante = int(cantidad)
    unidades_retiradas = 0
    costo_total = 0

    inventario.sort(key=lambda x: x["fecha_vencimiento"])

    for lote in inventario:
        if cantidad_restante <= 0:
            break

        retirar = min(lote["cantidad"], cantidad_restante)

        lote["cantidad"] -= retirar
        cantidad_restante -= retirar

        unidades_retiradas += retirar
        costo_total += retirar * lote["costo_unitario"]

    inventario[:] = [
        lote for lote in inventario
        if lote["cantidad"] > 0
    ]

    return unidades_retiradas, costo_total


def elegir_proveedor(producto_id, stock_final, demanda_esperada, vida_util):
    """
    Selecciona proveedor considerando costo, rapidez,
    inventario disponible y vida útil.
    """

    opciones = proveedores_por_producto[producto_id]

    cobertura_stock = (
        stock_final / demanda_esperada
        if demanda_esperada > 0
        else 0
    )

    # Productos muy perecibles: prioridad a rapidez
    if vida_util <= 2:
        return min(
            opciones,
            key=lambda x: x["dias_entrega_prometidos"]
        )

    # Si existe suficiente cobertura, podemos privilegiar costo
    if cobertura_stock >= 5:
        return min(
            opciones,
            key=lambda x: x["costo_unitario_proveedor"]
        )

    # Caso intermedio: equilibrio costo / rapidez
    costo_min = min(
        x["costo_unitario_proveedor"]
        for x in opciones
    )

    for opcion in opciones:
        opcion["_score"] = (
            opcion["dias_entrega_prometidos"] * 0.65
            + (opcion["costo_unitario_proveedor"] / costo_min) * 0.35
        )

    return min(opciones, key=lambda x: x["_score"])

In [76]:

rng = np.random.default_rng(2026)

operacion = []
reposiciones = []

pedido_num = 1

# Inventario por sucursal y producto
inventarios = {}

# Reposiciones que todavía vienen en camino
pedidos_pendientes = defaultdict(list)

# Última fecha en que se realizó una reposición
ultimo_pedido = {}


# ---------------------------------------------------------
# INVENTARIO INICIAL
# ---------------------------------------------------------

for sucursal in sucursales:

    for producto_id, producto in info_productos.items():

        demanda_inicial = (
            producto["demanda_base_dia"]
            * producto["factor_demanda"]
            * factor_demanda_sucursal[sucursal]
        )

        dias_stock_inicial = min(
            producto["vida_util_dias"],
            2
        )

        cantidad_inicial = max(
            1,
            int(round(
                demanda_inicial
                * dias_stock_inicial
            ))
        )

        inventarios[(sucursal, producto_id)] = [{
            "cantidad": cantidad_inicial,
            "costo_unitario": producto["costo_referencia"],
            "fecha_vencimiento":
                fechas[0]
                + pd.Timedelta(
                    days=producto["vida_util_dias"]
                )
        }]


# ---------------------------------------------------------
# SIMULACIÓN DIARIA
# ---------------------------------------------------------

for fecha in fechas:

    for sucursal in sucursales:

        for producto_id, producto in info_productos.items():

            clave = (
                sucursal,
                producto_id
            )

            inventario = inventarios[clave]


            # =================================================
            # 1. STOCK INICIAL
            # =================================================

            stock_inicial = sum(
                lote["cantidad"]
                for lote in inventario
            )


            # =================================================
            # 2. RECEPCIÓN DE REPOSICIONES
            # =================================================

            unidades_recibidas = 0

            entregas_hoy = pedidos_pendientes.pop(
                (
                    fecha,
                    sucursal,
                    producto_id
                ),
                []
            )

            for entrega in entregas_hoy:

                unidades_recibidas += (
                    entrega["unidades_recibidas"]
                )

                inventario.append({
                    "cantidad":
                        entrega["unidades_recibidas"],

                    "costo_unitario":
                        entrega["costo_unitario_proveedor"],

                    "fecha_vencimiento":
                        fecha
                        + pd.Timedelta(
                            days=producto["vida_util_dias"]
                        )
                })


            # Todo el inventario que debe gestionarse ese día
            unidades_disponibles = (
                stock_inicial
                + unidades_recibidas
            )


            # =================================================
            # 3. MERMA POR VENCIMIENTO
            # =================================================

            merma_vencimiento = 0
            costo_merma_vencimiento = 0

            lotes_vencidos = [
                lote
                for lote in inventario
                if lote["fecha_vencimiento"] < fecha
            ]

            for lote in lotes_vencidos:

                merma_vencimiento += (
                    lote["cantidad"]
                )

                costo_merma_vencimiento += (
                    lote["cantidad"]
                    * lote["costo_unitario"]
                )


            inventario[:] = [
                lote
                for lote in inventario
                if lote["fecha_vencimiento"] >= fecha
            ]


            # =================================================
            # 4. DEMANDA DEL DÍA
            # =================================================

            demanda_promedio = (
                producto["demanda_base_dia"]
                * producto["factor_demanda"]
                * factor_demanda_sucursal[sucursal]
                * factor_dia_semana[fecha.dayofweek]
            )


            demanda_real = max(
                0,
                int(round(
                    rng.normal(
                        demanda_promedio,
                        max(
                            1,
                            demanda_promedio * 0.18
                        )
                    )
                ))
            )


            stock_para_venta = sum(
                lote["cantidad"]
                for lote in inventario
            )


            # Las ventas nunca pueden superar
            # el inventario disponible
            ventas_posibles = min(
                demanda_real,
                stock_para_venta
            )


            unidades_vendidas, costo_ventas = (
                quitar_unidades_fifo(
                    inventario,
                    ventas_posibles
                )
            )


            # =================================================
            # 5. MERMA OPERACIONAL
            # =================================================

            stock_despues_ventas = sum(
                lote["cantidad"]
                for lote in inventario
            )


            prob_merma = (
                merma_base_categoria[
                    producto["categoria"]
                ]
                * producto["factor_merma"]
            )


            if stock_despues_ventas > 0:

                merma_operacional_objetivo = (
                    rng.binomial(
                        stock_despues_ventas,
                        min(
                            prob_merma,
                            0.20
                        )
                    )
                )

            else:

                merma_operacional_objetivo = 0


            (
                merma_operacional,
                costo_merma_operacional
            ) = quitar_unidades_fifo(
                inventario,
                merma_operacional_objetivo
            )


            merma_unidades = (
                merma_vencimiento
                + merma_operacional
            )


            costo_merma = (
                costo_merma_vencimiento
                + costo_merma_operacional
            )


            stock_final = sum(
                lote["cantidad"]
                for lote in inventario
            )


            # =================================================
            # 6. RESULTADOS FINANCIEROS
            # =================================================

            ingresos_netos = (
                unidades_vendidas
                * producto["precio_venta_neto"]
            )


            ganancia_bruta_neta_merma = (
                ingresos_netos
                - costo_ventas
                - costo_merma
            )


            # =================================================
            # 7. GUARDAR OPERACIÓN DEL DÍA
            # =================================================

            operacion.append({

                "fecha": fecha,

                "sucursal": sucursal,

                "producto_id":
                    producto_id,

                "producto":
                    producto["producto"],

                "categoria":
                    producto["categoria"],

                "vida_util_dias":
                    producto["vida_util_dias"],

                "stock_inicial":
                    stock_inicial,

                "unidades_recibidas":
                    unidades_recibidas,

                "unidades_disponibles":
                    unidades_disponibles,

                "unidades_vendidas":
                    unidades_vendidas,

                "merma_unidades":
                    merma_unidades,

                "stock_final":
                    stock_final,

                "precio_venta_neto":
                    producto["precio_venta_neto"],

                "costo_ventas":
                    round(
                        costo_ventas,
                        0
                    ),

                "costo_merma":
                    round(
                        costo_merma,
                        0
                    ),

                "ingresos_netos":
                    round(
                        ingresos_netos,
                        0
                    ),

                "ganancia_bruta_neta_merma":
                    round(
                        ganancia_bruta_neta_merma,
                        0
                    )
            })


            # =================================================
            # 8. PLANIFICACIÓN DE REPOSICIÓN
            # =================================================

            demanda_esperada = max(
                1,
                demanda_promedio
            )


            # Cantidad que ya fue pedida
            # pero todavía no ha llegado
            cantidad_pendiente = 0


            for (
                fecha_entrega,
                suc_p,
                prod_p
            ), entregas in pedidos_pendientes.items():

                if (
                    suc_p == sucursal
                    and prod_p == producto_id
                    and fecha_entrega > fecha
                ):

                    cantidad_pendiente += sum(
                        x["unidades_recibidas"]
                        for x in entregas
                    )


            inventario_en_posicion = (
                stock_final
                + cantidad_pendiente
            )


            # Selección del proveedor
            proveedor = elegir_proveedor(
                producto_id,
                inventario_en_posicion,
                demanda_esperada,
                producto["vida_util_dias"]
            )


            lead_time = proveedor[
                "dias_entrega_prometidos"
            ]


            # -------------------------------------------------
            # Frecuencia de revisión del inventario
            # según vida útil
            # -------------------------------------------------

            if producto["vida_util_dias"] == 1:

                intervalo_revision = 1

            elif producto["vida_util_dias"] == 2:

                intervalo_revision = 2

            elif producto["vida_util_dias"] == 3:

                intervalo_revision = 3

            else:

                intervalo_revision = 4


            # -------------------------------------------------
            # Verificar cuándo hicimos el último pedido
            # -------------------------------------------------

            fecha_ultimo_pedido = (
                ultimo_pedido.get(clave)
            )


            puede_pedir = (

                fecha_ultimo_pedido is None

                or

                (
                    fecha
                    - fecha_ultimo_pedido
                ).days >= intervalo_revision
            )


            # -------------------------------------------------
            # Punto de reposición
            #
            # Demanda necesaria para cubrir el tiempo
            # del proveedor + 1 día de seguridad
            # -------------------------------------------------

            stock_seguridad = (
                demanda_esperada * 1
            )


            punto_reposicion = (
                demanda_esperada
                * lead_time
                + stock_seguridad
            )


            # -------------------------------------------------
            # Inventario objetivo después de pedir
            # -------------------------------------------------

            dias_post_entrega = min(
                producto["vida_util_dias"],
                intervalo_revision + 1
            )


            inventario_objetivo = (

                demanda_esperada
                * (
                    lead_time
                    + dias_post_entrega
                )

                * producto[
                    "factor_reposicion"
                ]

                * factor_reposicion_sucursal[
                    sucursal
                ]
            )


            necesidad_reposicion = int(
                round(
                    inventario_objetivo
                    - inventario_en_posicion
                )
            )


            # =================================================
            # 9. GENERAR PEDIDO SOLO SI ES NECESARIO
            # =================================================

            if (
                puede_pedir
                and inventario_en_posicion
                    <= punto_reposicion
                and necesidad_reposicion > 0
            ):


                unidades_solicitadas = (
                    necesidad_reposicion
                )


                perfil = perfil_proveedores[
                    proveedor["proveedor"]
                ]


                # ¿El proveedor se atrasa?
                tiene_retraso = (
                    rng.random()
                    < perfil["prob_retraso"]
                )


                if tiene_retraso:

                    dias_retraso = int(
                        rng.integers(
                            1,
                            4
                        )
                    )

                else:

                    dias_retraso = 0


                fecha_prometida = (
                    fecha
                    + pd.Timedelta(
                        days=lead_time
                    )
                )


                fecha_entrega_real = (
                    fecha_prometida
                    + pd.Timedelta(
                        days=dias_retraso
                    )
                )


                # Pequeña variación del costo
                # entre pedidos
                costo_pedido = int(
                    round(
                        (
                            proveedor[
                                "costo_unitario_proveedor"
                            ]
                            * rng.uniform(
                                0.98,
                                1.02
                            )
                        )
                        / 10
                    )
                    * 10
                )


                # Algunos pedidos pueden llegar
                # ligeramente incompletos
                porcentaje_recibido = (
                    rng.uniform(
                        0.92,
                        1.00
                    )
                )


                unidades_recibidas_pedido = max(
                    1,
                    int(round(
                        unidades_solicitadas
                        * porcentaje_recibido
                    ))
                )


                registro_pedido = {

                    "pedido_id":
                        f"PED{pedido_num:06d}",

                    "fecha_pedido":
                        fecha,

                    "sucursal":
                        sucursal,

                    "producto_id":
                        producto_id,

                    "producto":
                        producto["producto"],

                    "proveedor":
                        proveedor["proveedor"],

                    "unidades_solicitadas":
                        unidades_solicitadas,

                    "unidades_recibidas":
                        unidades_recibidas_pedido,

                    "costo_unitario_proveedor":
                        costo_pedido,

                    "fecha_prometida":
                        fecha_prometida,

                    "fecha_entrega_real":
                        fecha_entrega_real,

                    "dias_entrega":
                        (
                            fecha_entrega_real
                            - fecha
                        ).days,

                    "dias_retraso":
                        dias_retraso
                }


                reposiciones.append(
                    registro_pedido
                )


                pedidos_pendientes[
                    (
                        fecha_entrega_real,
                        sucursal,
                        producto_id
                    )
                ].append(
                    registro_pedido
                )


                ultimo_pedido[
                    clave
                ] = fecha


                pedido_num += 1

In [77]:
df = pd.DataFrame(operacion)

df_reposiciones = pd.DataFrame(
    reposiciones
)

print(
    "Operación diaria:",
    df.shape
)

print(
    "Líneas de reposición:",
    df_reposiciones.shape
)

display(df.head())

display(
    df_reposiciones.head()
)

Operación diaria: (87600, 17)
Líneas de reposición: (50667, 13)


,fecha,sucursal,producto_id,producto,categoria,vida_util_dias,stock_inicial,unidades_recibidas,unidades_disponibles,unidades_vendidas,merma_unidades,stock_final,precio_venta_neto,costo_ventas,costo_merma,ingresos_netos,ganancia_bruta_neta_merma
0,2026-01-01,Ñuñoa,P001,Croissant mantequilla,Panadería,2,59,0,59,26,0,33,3700,49400,0,96200,46800
1,2026-01-01,Ñuñoa,P002,Pan de chocolate,Panadería,2,85,0,85,41,2,42,3500,57810,2820,143500,82870
2,2026-01-01,Ñuñoa,P003,Marraqueta gourmet,Panadería,2,34,0,34,20,1,13,3600,42000,2100,72000,27900
3,2026-01-01,Ñuñoa,P004,Baguette integral,Panadería,1,30,0,30,28,0,2,2900,40600,0,81200,40600
4,2026-01-01,Ñuñoa,P005,Pan de masa madre,Panadería,2,53,0,53,25,0,28,3300,61750,0,82500,20750


,pedido_id,fecha_pedido,sucursal,producto_id,producto,proveedor,unidades_solicitadas,unidades_recibidas,costo_unitario_proveedor,fecha_prometida,fecha_entrega_real,dias_entrega,dias_retraso
0,PED000001,2026-01-01,Ñuñoa,P001,Croissant mantequilla,Proveedor D,58,55,2060,2026-01-02,2026-01-02,1,0
1,PED000002,2026-01-01,Ñuñoa,P002,Pan de chocolate,Proveedor A,133,126,1430,2026-01-03,2026-01-03,2,0
2,PED000003,2026-01-01,Ñuñoa,P003,Marraqueta gourmet,Proveedor D,39,37,2230,2026-01-02,2026-01-02,1,0
3,PED000004,2026-01-01,Ñuñoa,P004,Baguette integral,Proveedor D,61,57,1570,2026-01-02,2026-01-02,1,0
4,PED000005,2026-01-01,Ñuñoa,P005,Pan de masa madre,Proveedor A,81,77,2530,2026-01-03,2026-01-03,2,0


In [78]:
print(
    "Promedio de líneas de reposición por día:",
    round(
        len(df_reposiciones) / df_reposiciones["fecha_pedido"].nunique(),
        2
    )
)

print(
    "\nPedidos duplicados mismo día / sucursal / producto:",
    df_reposiciones.duplicated(
        subset=["fecha_pedido", "sucursal", "producto_id"]
    ).sum()
)

reposiciones_por_producto = (
    df_reposiciones
    .groupby(["producto_id", "producto", "sucursal"])
    .size()
    .reset_index(name="reposiciones_anuales")
)

display(
    reposiciones_por_producto[
        "reposiciones_anuales"
    ].describe()
)

display(
    reposiciones_por_producto
    .sort_values(
        "reposiciones_anuales",
        ascending=False
    )
    .head(10)
)

Promedio de líneas de reposición por día: 138.81

Pedidos duplicados mismo día / sucursal / producto: 0


,reposiciones_anuales
count,240.00000
mean,211.11250
std,105.73813
min,45.00000
25%,120.00000
50%,178.00000
75%,346.25000
max,365.00000


,producto_id,producto,sucursal,reposiciones_anuales
13,P004,Baguette integral,Providencia,365
239,P060,Wrap caliente,Ñuñoa,365
217,P055,Bowl pollo,Providencia,365
233,P059,Pastel de choclo,Providencia,365
145,P037,Ensalada César,Providencia,365
165,P042,Ensalada atún,Providencia,365
174,P044,Ensalada cous cous,Santiago Centro,365
139,P035,Club sandwich,Ñuñoa,365
130,P033,Bagel pavo,Santiago Centro,364
14,P004,Baguette integral,Santiago Centro,364


## Validación de calidad de datos

Antes de realizar el análisis se revisará la calidad de ambas tablas para detectar posibles problemas que puedan afectar los resultados.

Se comprobarán:

- valores faltantes,
- registros duplicados,
- tipos de datos,
- valores negativos o inconsistentes,
- coherencia entre inventario disponible, ventas, merma y stock final.

In [79]:
print("=== OPERACIÓN DIARIA ===")

print("\nValores nulos:")
display(df.isnull().sum())

print("\nDuplicados:")
print(
    df.duplicated(
        subset=["fecha", "sucursal", "producto_id"]
    ).sum()
)

print("\nTipos de datos:")
display(df.dtypes)


print("\n=== REPOSICIONES ===")

print("\nValores nulos:")
display(df_reposiciones.isnull().sum())

print("\nDuplicados:")
print(
    df_reposiciones.duplicated(
        subset=["pedido_id"]
    ).sum()
)

print("\nTipos de datos:")
display(df_reposiciones.dtypes)

=== OPERACIÓN DIARIA ===

Valores nulos:


,0
fecha,0
sucursal,0
producto_id,0
producto,0
categoria,0
vida_util_dias,0
stock_inicial,0
unidades_recibidas,0
unidades_disponibles,0
unidades_vendidas,0



Duplicados:
0

Tipos de datos:


,0
fecha,datetime64[ns]
sucursal,object
producto_id,object
producto,object
categoria,object
vida_util_dias,int64
stock_inicial,int64
unidades_recibidas,int64
unidades_disponibles,int64
unidades_vendidas,int64



=== REPOSICIONES ===

Valores nulos:


,0
pedido_id,0
fecha_pedido,0
sucursal,0
producto_id,0
producto,0
proveedor,0
unidades_solicitadas,0
unidades_recibidas,0
costo_unitario_proveedor,0
fecha_prometida,0



Duplicados:
0

Tipos de datos:


,0
pedido_id,object
fecha_pedido,datetime64[ns]
sucursal,object
producto_id,object
producto,object
proveedor,object
unidades_solicitadas,int64
unidades_recibidas,int64
costo_unitario_proveedor,int64
fecha_prometida,datetime64[ns]


In [80]:
validaciones = {
    "stock_inicial_negativo":
        (df["stock_inicial"] < 0).sum(),

    "ventas_negativas":
        (df["unidades_vendidas"] < 0).sum(),

    "merma_negativa":
        (df["merma_unidades"] < 0).sum(),

    "stock_final_negativo":
        (df["stock_final"] < 0).sum(),

    "merma_superior_disponible":
        (
            df["merma_unidades"]
            > df["unidades_disponibles"]
        ).sum(),

    "ventas_superiores_disponible":
        (
            df["unidades_vendidas"]
            > df["unidades_disponibles"]
        ).sum()
}

pd.Series(validaciones)

,0
stock_inicial_negativo,0
ventas_negativas,0
merma_negativa,0
stock_final_negativo,0
merma_superior_disponible,0
ventas_superiores_disponible,0


### Resultado de la validación

No se detectaron valores faltantes, registros duplicados ni inconsistencias numéricas relevantes.

Las cantidades de inventario, ventas y merma presentan valores válidos y las ventas o mermas no superan las unidades disponibles.

Por lo tanto, las bases se consideran aptas para continuar con el análisis.

## 1. Inventario y merma

El primer análisis busca identificar productos con señales de sobreproducción, baja rotación y merma recurrente.

Para evitar conclusiones basadas únicamente en cantidades absolutas, se utilizarán indicadores relativos que permitan comparar productos con diferentes niveles de inventario y demanda:

- **Tasa de merma:** proporción de las unidades disponibles que terminaron en merma.
- **Sell-through:** proporción del inventario disponible que logró venderse.
- **Stock remanente:** proporción del inventario disponible que permaneció en stock al finalizar el día.

Una combinación de alta merma, bajo sell-through y stock remanente elevado puede indicar problemas de planificación del inventario.

In [81]:
inventario_producto = (
    df.groupby(
        ["producto_id", "producto", "categoria"]
    )
    .agg(
        unidades_disponibles=("unidades_disponibles", "sum"),
        unidades_vendidas=("unidades_vendidas", "sum"),
        merma_unidades=("merma_unidades", "sum"),
        stock_final_total=("stock_final", "sum"),
        stock_final_promedio=("stock_final", "mean"),
        registros_con_merma=(
            "merma_unidades",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

inventario_producto["tasa_merma"] = (
    inventario_producto["merma_unidades"]
    / inventario_producto["unidades_disponibles"]
    * 100
)

inventario_producto["sell_through"] = (
    inventario_producto["unidades_vendidas"]
    / inventario_producto["unidades_disponibles"]
    * 100
)

inventario_producto["stock_remanente_pct"] = (
    inventario_producto["stock_final_total"]
    / inventario_producto["unidades_disponibles"]
    * 100
)

inventario_producto = (
    inventario_producto
    .sort_values(
        "tasa_merma",
        ascending=False
    )
)

display(
    inventario_producto.head(10)
)

,producto_id,producto,categoria,unidades_disponibles,unidades_vendidas,merma_unidades,stock_final_total,stock_final_promedio,registros_con_merma,tasa_merma,sell_through,stock_remanente_pct
19,P020,Muffin arándanos,Pastelería,45412,13161,4639,27612,18.912329,789,10.215362,28.981327,60.803312
47,P048,Ensalada griega,Ensaladas,18186,8151,1676,8359,5.725342,629,9.215880,44.820191,45.963928
13,P014,Torta chocolate,Pastelería,88132,26955,7848,53329,36.526712,1033,8.904825,30.584805,60.510371
45,P046,Ensalada garbanzos,Ensaladas,16911,7546,1325,8040,5.506849,644,7.835137,44.621844,47.543019
39,P040,Ensalada caprese,Ensaladas,20660,9244,1589,9827,6.730822,678,7.691191,44.743466,47.565344
37,P038,Ensalada quinoa,Ensaladas,23316,8025,1729,13562,9.289041,521,7.415509,34.418425,58.166066
40,P041,Ensalada pollo,Ensaladas,11609,4074,793,6742,4.617808,409,6.830907,35.093462,58.075631
49,P050,Quiche espinaca,Platos preparados,19654,10267,1230,8157,5.586986,498,6.258268,52.238730,41.503002
38,P039,Ensalada mediterránea,Ensaladas,33864,17846,2099,13919,9.533562,599,6.198323,52.699031,41.102646
44,P045,Ensalada pasta,Ensaladas,23523,8313,1456,13754,9.420548,552,6.189687,35.339880,58.470433


### Hallazgo

Los productos con mayor tasa de merma también presentan señales de exceso de inventario y baja rotación.

Galleta NY chocolate y Torta de chocolate muestran una combinación de merma elevada y stock remanente importante, mientras que Muffin arándanos destaca por mantener una mayor proporción de inventario sin vender.

Estos resultados sugieren revisar los niveles y frecuencia de reposición de estos productos antes de reducir precios o eliminarlos, ya que el problema podría estar asociado a una planificación de inventario superior a la demanda real.

### Productos de baja rotación

Para identificar productos que podrían estar manteniéndose en inventario por encima de su demanda real, se analizarán distintas señales de baja rotación:

- días con ventas iguales a cero,
- días con ventas iguales o inferiores a 10 unidades,
- frecuencia con la que el producto termina el día con stock disponible,
- sell-through,
- stock final promedio.

Un producto con ventas bajas de forma recurrente y stock remanente frecuente puede indicar una reposición superior a la demanda real.

In [82]:
baja_rotacion = (
    df.groupby(
        ["producto_id", "producto", "categoria"]
    )
    .agg(
        dias_totales=("fecha", "count"),
        dias_venta_cero=(
            "unidades_vendidas",
            lambda x: (x == 0).sum()
        ),
        dias_venta_baja=(
            "unidades_vendidas",
            lambda x: (x <= 10).sum()
        ),
        dias_con_stock_final=(
            "stock_final",
            lambda x: (x > 0).sum()
        ),
        unidades_disponibles=("unidades_disponibles", "sum"),
        unidades_vendidas=("unidades_vendidas", "sum"),
        stock_final_promedio=("stock_final", "mean")
    )
    .reset_index()
)

baja_rotacion["pct_dias_venta_cero"] = (
    baja_rotacion["dias_venta_cero"]
    / baja_rotacion["dias_totales"]
    * 100
)

baja_rotacion["pct_dias_venta_baja"] = (
    baja_rotacion["dias_venta_baja"]
    / baja_rotacion["dias_totales"]
    * 100
)

baja_rotacion["pct_dias_con_stock"] = (
    baja_rotacion["dias_con_stock_final"]
    / baja_rotacion["dias_totales"]
    * 100
)

baja_rotacion["sell_through"] = (
    baja_rotacion["unidades_vendidas"]
    / baja_rotacion["unidades_disponibles"]
    * 100
)

baja_rotacion = (
    baja_rotacion
    .sort_values(
        [
            "pct_dias_venta_baja",
            "pct_dias_con_stock"
        ],
        ascending=[False, False]
    )
)

display(
    baja_rotacion.head(10)
)

,producto_id,producto,categoria,dias_totales,dias_venta_cero,dias_venta_baja,dias_con_stock_final,unidades_disponibles,unidades_vendidas,stock_final_promedio,pct_dias_venta_cero,pct_dias_venta_baja,pct_dias_con_stock,sell_through
44,P045,Ensalada pasta,Ensaladas,1460,34,1460,1397,23523,8313,9.420548,2.328767,100.000000,95.684932,35.339880
45,P046,Ensalada garbanzos,Ensaladas,1460,11,1460,1369,16911,7546,5.506849,0.753425,100.000000,93.767123,44.621844
40,P041,Ensalada pollo,Ensaladas,1460,63,1460,1347,11609,4074,4.617808,4.315068,100.000000,92.260274,35.093462
37,P038,Ensalada quinoa,Ensaladas,1460,68,1460,1326,23316,8025,9.289041,4.657534,100.000000,90.821918,34.418425
47,P048,Ensalada griega,Ensaladas,1460,33,1459,1306,18186,8151,5.725342,2.260274,99.931507,89.452055,44.820191
39,P040,Ensalada caprese,Ensaladas,1460,10,1453,1379,20660,9244,6.730822,0.684932,99.520548,94.452055,44.743466
49,P050,Quiche espinaca,Platos preparados,1460,21,1406,1259,19654,10267,5.586986,1.438356,96.301370,86.232877,52.238730
50,P051,Lasaña boloñesa,Platos preparados,1460,56,1290,1263,28226,11593,10.437671,3.835616,88.356164,86.506849,41.072061
22,P023,Alfajor,Pastelería,1460,37,1286,1326,39974,11755,18.578767,2.534247,88.082192,90.821918,29.406614
42,P043,Ensalada veggie,Ensaladas,1460,49,1263,1286,28845,11808,10.715068,3.356164,86.506849,88.082192,40.936037


### Hallazgo

A nivel agregado, la baja rotación se concentra principalmente en productos de la categoría Ensaladas y en Muffin arándanos.

Muffin arándanos presenta un sell-through cercano al 20%, una elevada frecuencia de días sin ventas y stock disponible al cierre en gran parte de los registros.

Ensalada pasta y Ensalada palta también presentan sell-through reducido y una alta frecuencia de inventario remanente. Además, productos como Ensalada veggie, Ensalada cous cous, Ensalada quinoa y Ensalada garbanzos registran ventas bajas de forma recurrente.

Sin embargo, estos resultados corresponden al comportamiento agregado de todas las sucursales, por lo que es necesario revisar cada sucursal por separado antes de recomendar cambios en los niveles de reposición.

## 2. Quiebres potenciales de stock

Para detectar posibles quiebres de stock no se considerará únicamente que el inventario termine en cero.

Se utilizará como referencia el promedio de ventas de los 28 días anteriores para cada combinación producto-sucursal. El día actual no se incluirá dentro de su propio promedio.

Se considerará una señal de posible quiebre cuando:

- exista un historial completo de 28 días,
- el producto termine el día sin stock,
- y las unidades vendidas sean inferiores a la demanda esperada.

La diferencia entre la demanda esperada y las ventas observadas se utilizará como una estimación de demanda potencialmente no atendida. Este indicador representa una aproximación y no una medición directa de ventas perdidas.

In [83]:
df = df.sort_values(
    ["sucursal", "producto_id", "fecha"]
)

df["demanda_esperada_28d"] = (
    df.groupby(
        ["sucursal", "producto_id"]
    )["unidades_vendidas"]
    .transform(
        lambda x: x.shift(1)
                   .rolling(
                       window=28,
                       min_periods=28
                   )
                   .mean()
    )
)

df["quiebre_potencial"] = (
    (df["demanda_esperada_28d"].notna())
    &
    (df["stock_final"] == 0)
    &
    (df["unidades_vendidas"] < df["demanda_esperada_28d"])
).astype(int)

df["demanda_no_atendida_est"] = np.where(
    df["quiebre_potencial"] == 1,
    df["demanda_esperada_28d"] - df["unidades_vendidas"],
    0
)

In [84]:
df = df.sort_values(
    ["sucursal", "producto_id", "fecha"]
)

df["demanda_esperada_28d"] = (
    df.groupby(
        ["sucursal", "producto_id"]
    )["unidades_vendidas"]
    .transform(
        lambda x: x.shift(1)
                   .rolling(28, min_periods=7)
                   .mean()
    )
)

df["quiebre_potencial"] = (
    (df["stock_final"] == 0)
    &
    (df["unidades_vendidas"] < df["demanda_esperada_28d"])
)

df["demanda_no_atendida_est"] = np.where(
    df["quiebre_potencial"],
    df["demanda_esperada_28d"] - df["unidades_vendidas"],
    0
)

In [85]:
quiebres_resumen = (
    df.groupby(
        ["sucursal", "producto_id", "producto", "categoria"]
    )
    .agg(
        dias_analizados=(
            "demanda_esperada_28d",
            "count"
        ),
        dias_quiebre_potencial=(
            "quiebre_potencial",
            "sum"
        ),
        demanda_no_atendida_est=(
            "demanda_no_atendida_est",
            "sum"
        )
    )
    .reset_index()
)

quiebres_resumen["pct_dias_quiebre"] = (
    quiebres_resumen["dias_quiebre_potencial"]
    / quiebres_resumen["dias_analizados"]
    * 100
)

quiebres_resumen = quiebres_resumen.sort_values(
    "pct_dias_quiebre",
    ascending=False
)

display(quiebres_resumen.head(15))

,sucursal,producto_id,producto,categoria,dias_analizados,dias_quiebre_potencial,demanda_no_atendida_est,pct_dias_quiebre
20,Maipú,P021,Galleta NY chocolate,Pastelería,358,99,957.029320,27.653631
14,Maipú,P015,Tiramisú,Pastelería,358,98,1133.222758,27.374302
155,Santiago Centro,P036,Croissant relleno,Sándwiches,358,85,739.929930,23.743017
17,Maipú,P018,Pie de limón,Pastelería,358,73,824.804277,20.391061
150,Santiago Centro,P031,Wrap mediterráneo,Sándwiches,358,69,713.704113,19.273743
215,Ñuñoa,P036,Croissant relleno,Sándwiches,358,69,474.057692,19.273743
95,Providencia,P036,Croissant relleno,Sándwiches,358,68,662.848255,18.994413
153,Santiago Centro,P034,Ciabatta italiana,Sándwiches,358,56,916.317424,15.642458
90,Providencia,P031,Wrap mediterráneo,Sándwiches,358,55,494.112963,15.363128
144,Santiago Centro,P025,Sándwich ave palta,Sándwiches,358,55,306.860341,15.363128


In [86]:
muffin_sucursal = (
    df[df["producto"] == "Muffin arándanos"]
    .groupby("sucursal")
    .agg(
        unidades_disponibles=("unidades_disponibles", "sum"),
        unidades_vendidas=("unidades_vendidas", "sum"),
        merma_unidades=("merma_unidades", "sum"),
        stock_final_total=("stock_final", "sum"),
        dias_con_stock=(
            "stock_final",
            lambda x: (x > 0).sum()
        ),
        dias_quiebre_potencial=(
            "quiebre_potencial",
            "sum"
        )
    )
    .reset_index()
)

muffin_sucursal["sell_through"] = (
    muffin_sucursal["unidades_vendidas"]
    / muffin_sucursal["unidades_disponibles"]
    * 100
)

muffin_sucursal["tasa_merma"] = (
    muffin_sucursal["merma_unidades"]
    / muffin_sucursal["unidades_disponibles"]
    * 100
)

muffin_sucursal["pct_dias_con_stock"] = (
    muffin_sucursal["dias_con_stock"]
    / 365
    * 100
)

display(muffin_sucursal)

,sucursal,unidades_disponibles,unidades_vendidas,merma_unidades,stock_final_total,dias_con_stock,dias_quiebre_potencial,sell_through,tasa_merma,pct_dias_con_stock
0,Maipú,10185,2542,1287,6356,327,36,24.958272,12.636230,89.589041
1,Providencia,11961,3628,1119,7214,350,14,30.331912,9.355405,95.890411
2,Santiago Centro,11739,3570,1141,7028,333,25,30.411449,9.719738,91.232877
3,Ñuñoa,11527,3421,1092,7014,343,16,29.678147,9.473410,93.972603


### Hallazgo por sucursal: Muffin arándanos

El comportamiento de Muffin arándanos varía entre sucursales.

Santiago Centro presenta la mayor persistencia de inventario remanente, terminando con stock disponible en aproximadamente el 93% de los días analizados.

Maipú, en cambio, registra 97 días con señales de posible quiebre de stock, equivalentes a cerca del 27% del período. Sin embargo, también mantiene inventario al cierre en más del 73% de los días, junto con el menor sell-through y la mayor tasa de merma entre las sucursales.

Esto indica que el problema no parece limitarse a una falta de inventario, sino que podría estar relacionado con la cantidad y el momento de las reposiciones. Es decir, pueden existir períodos de sobreabastecimiento y otros de insuficiencia dentro de una misma sucursal.

### Hallazgo: posibles quiebres de stock

Los posibles quiebres de stock no se concentran en un único producto.

Muffin arándanos en Maipú presenta la mayor frecuencia, con 97 días de posible quiebre (26,58%). Donut glaseada en la misma sucursal también registra una frecuencia elevada.

Además, la categoría Sándwiches presenta un patrón recurrente: Sándwich roast beef y Bagel salmón aparecen entre los productos con mayor frecuencia de posibles quiebres en varias sucursales.

Esto sugiere que existen problemas de reposición específicos por producto y sucursal, y que algunos casos podrían responder a una planificación insuficiente frente a la demanda.

In [87]:
reposicion_resumen = (
    df_reposiciones
    .groupby(
        ["sucursal", "producto_id", "producto"]
    )
    .agg(
        cantidad_pedidos=("pedido_id", "count"),
        unidades_solicitadas=("unidades_solicitadas", "sum"),
        unidades_recibidas=("unidades_recibidas", "sum"),
        dias_entrega_promedio=("dias_entrega", "mean"),
        dias_retraso_promedio=("dias_retraso", "mean"),
        pct_pedidos_atrasados=(
            "dias_retraso",
            lambda x: (x > 0).mean() * 100
        )
    )
    .reset_index()
)

reposicion_resumen["pct_unidades_recibidas"] = (
    reposicion_resumen["unidades_recibidas"]
    / reposicion_resumen["unidades_solicitadas"]
    * 100
)

quiebres_reposicion = (
    quiebres_resumen
    .merge(
        reposicion_resumen,
        on=["sucursal", "producto_id", "producto"],
        how="left"
    )
    .sort_values(
        "pct_dias_quiebre",
        ascending=False
    )
)

display(
    quiebres_reposicion[
        [
            "sucursal",
            "producto",
            "pct_dias_quiebre",
            "demanda_no_atendida_est",
            "cantidad_pedidos",
            "dias_entrega_promedio",
            "pct_pedidos_atrasados",
            "pct_unidades_recibidas"
        ]
    ].head(15)
)

,sucursal,producto,pct_dias_quiebre,demanda_no_atendida_est,cantidad_pedidos,dias_entrega_promedio,pct_pedidos_atrasados,pct_unidades_recibidas
0,Maipú,Galleta NY chocolate,27.653631,957.029320,45,4.422222,24.444444,95.868852
1,Maipú,Tiramisú,27.374302,1133.222758,45,4.755556,37.777778,95.489033
2,Santiago Centro,Croissant relleno,23.743017,739.929930,183,2.136612,7.650273,95.871192
3,Maipú,Pie de limón,20.391061,824.804277,50,3.400000,14.000000,95.400807
4,Santiago Centro,Wrap mediterráneo,19.273743,713.704113,358,2.187151,8.938547,96.252512
5,Ñuñoa,Croissant relleno,19.273743,474.057692,182,2.164835,6.593407,95.843936
6,Providencia,Croissant relleno,18.994413,662.848255,183,2.202186,8.743169,95.904389
7,Santiago Centro,Ciabatta italiana,15.642458,916.317424,358,1.195531,8.379888,96.245984
8,Providencia,Wrap mediterráneo,15.363128,494.112963,357,2.176471,7.843137,95.976345
9,Santiago Centro,Sándwich ave palta,15.363128,306.860341,182,1.126374,6.043956,95.899894


### Hallazgo: reposición y posibles causas de quiebre

Los resultados muestran que los posibles quiebres no parecen tener una única causa.

En Maipú, Muffin arándanos y Donut glaseada combinan una elevada frecuencia de posibles quiebres con tiempos de entrega superiores a 4 días y porcentajes de pedidos atrasados cercanos o superiores al 18%. Esto sugiere que los tiempos de reposición y el incumplimiento de plazos podrían estar contribuyendo al problema.

En cambio, productos como Sándwich roast beef y Bagel salmón presentan frecuentes señales de quiebre pese a tiempos de entrega cercanos a 2 días, bajos niveles de atraso y porcentajes de unidades recibidas próximos al 96%.

En estos casos, el problema parece estar más relacionado con la planificación de las cantidades y frecuencia de reposición que con el desempeño del proveedor.

Estos resultados muestran asociaciones y no permiten atribuir causalidad directa a un proveedor específico.

In [88]:
rentabilidad_producto = (
    df.groupby(
        ["producto_id", "producto", "categoria"]
    )
    .agg(
        unidades_vendidas=("unidades_vendidas", "sum"),
        ingresos_netos=("ingresos_netos", "sum"),
        costo_ventas=("costo_ventas", "sum"),
        costo_merma=("costo_merma", "sum"),
        ganancia=("ganancia_bruta_neta_merma", "sum")
    )
    .reset_index()
)

rentabilidad_producto["margen_pct"] = (
    rentabilidad_producto["ganancia"]
    / rentabilidad_producto["ingresos_netos"]
    * 100
)

rentabilidad_producto = rentabilidad_producto.sort_values(
    "ganancia",
    ascending=False
)

display(rentabilidad_producto.head(15))

,producto_id,producto,categoria,unidades_vendidas,ingresos_netos,costo_ventas,costo_merma,ganancia,margen_pct
31,P032,Bagel salmón,Sándwiches,38784,213312000,86458520,7643150,119210330,55.885431
1,P002,Pan de chocolate,Panadería,58827,205894500,83513380,9969090,112412030,54.596908
53,P054,Pasta pomodoro,Platos preparados,24851,211233500,93925840,10418480,106889180,50.602381
33,P034,Ciabatta italiana,Sándwiches,39117,293377500,185304110,3537590,104535800,35.631840
52,P053,Pasta pesto,Platos preparados,29640,189696000,84281430,9486260,95928310,50.569495
25,P026,Sándwich jamón queso,Sándwiches,38436,199867200,103777900,2317290,93772010,46.917158
57,P058,Sopa tomate,Platos preparados,26550,196470000,95472450,10409530,90588020,46.107813
8,P009,Pan de queso,Panadería,42858,158574600,67695080,5419680,85459840,53.892515
35,P036,Croissant relleno,Sándwiches,26045,171897000,89615990,2474660,79806350,46.426843
51,P052,Lasaña vegetariana,Platos preparados,22293,169426800,82679990,7586470,79160340,46.722443


In [89]:
rentabilidad_vista = rentabilidad_producto.copy()

precio_promedio = (
    df.groupby("producto_id")["precio_venta_neto"]
    .mean()
)

rentabilidad_vista["precio_venta_promedio"] = (
    rentabilidad_vista["producto_id"].map(precio_promedio)
)

rentabilidad_vista["ganancia_millones"] = (
    rentabilidad_vista["ganancia"] / 1_000_000
)

display(
    rentabilidad_vista[
        [
            "producto",
            "unidades_vendidas",
            "precio_venta_promedio",
            "ganancia_millones",
            "margen_pct"
        ]
    ]
    .head(15)
    .round({
        "precio_venta_promedio": 0,
        "ganancia_millones": 1,
        "margen_pct": 1
    })
)

,producto,unidades_vendidas,precio_venta_promedio,ganancia_millones,margen_pct
31,Bagel salmón,38784,5500.0,119.2,55.9
1,Pan de chocolate,58827,3500.0,112.4,54.6
53,Pasta pomodoro,24851,8500.0,106.9,50.6
33,Ciabatta italiana,39117,7500.0,104.5,35.6
52,Pasta pesto,29640,6400.0,95.9,50.6
25,Sándwich jamón queso,38436,5200.0,93.8,46.9
57,Sopa tomate,26550,7400.0,90.6,46.1
8,Pan de queso,42858,3700.0,85.5,53.9
35,Croissant relleno,26045,6600.0,79.8,46.4
51,Lasaña vegetariana,22293,7600.0,79.2,46.7


In [90]:
display(
    rentabilidad_producto[
        ["producto", "unidades_vendidas", "ganancia", "margen_pct"]
    ]
    .sort_values("margen_pct", ascending=False)
    .head(10)
    .round({"margen_pct": 1})
)

,producto,unidades_vendidas,ganancia,margen_pct
31,Bagel salmón,38784,119210330,55.9
1,Pan de chocolate,58827,112412030,54.6
8,Pan de queso,42858,85459840,53.9
11,Pan pita,34403,49661620,53.5
16,Brownie,15068,41184340,51.6
53,Pasta pomodoro,24851,106889180,50.6
52,Pasta pesto,29640,95928310,50.6
32,Bagel pavo,16448,44094700,49.6
24,Sándwich ave palta,19781,69119270,49.2
41,Ensalada atún,13659,40207660,49.1


In [91]:
mediana_ventas = rentabilidad_producto["unidades_vendidas"].median()
mediana_ganancia = rentabilidad_producto["ganancia"].median()

condiciones = [
    (
        (rentabilidad_producto["unidades_vendidas"] >= mediana_ventas) &
        (rentabilidad_producto["ganancia"] >= mediana_ganancia)
    ),
    (
        (rentabilidad_producto["unidades_vendidas"] >= mediana_ventas) &
        (rentabilidad_producto["ganancia"] < mediana_ganancia)
    ),
    (
        (rentabilidad_producto["unidades_vendidas"] < mediana_ventas) &
        (rentabilidad_producto["ganancia"] >= mediana_ganancia)
    )
]

categorias = [
    "Estrella",
    "Revisar margen/costos",
    "Potencial"
]

rentabilidad_producto["clasificacion"] = np.select(
    condiciones,
    categorias,
    default="Revisar continuidad"
)

display(
    rentabilidad_producto[
        [
            "producto",
            "unidades_vendidas",
            "ganancia",
            "margen_pct",
            "clasificacion"
        ]
    ]
    .sort_values(
        ["clasificacion", "ganancia"],
        ascending=[True, False]
    )
)

,producto,unidades_vendidas,ganancia,margen_pct,clasificacion
31,Bagel salmón,38784,119210330,55.885431,Estrella
1,Pan de chocolate,58827,112412030,54.596908,Estrella
53,Pasta pomodoro,24851,106889180,50.602381,Estrella
33,Ciabatta italiana,39117,104535800,35.631840,Estrella
52,Pasta pesto,29640,95928310,50.569495,Estrella
25,Sándwich jamón queso,38436,93772010,46.917158,Estrella
57,Sopa tomate,26550,90588020,46.107813,Estrella
8,Pan de queso,42858,85459840,53.892515,Estrella
35,Croissant relleno,26045,79806350,46.426843,Estrella
29,Wrap pollo,25094,67495660,46.374363,Estrella


### Hallazgo: rentabilidad y productos estrella

Bagel pavo lidera la ganancia total del portafolio, mientras que Carrot cake presenta el mayor margen porcentual.

La clasificación basada en ventas y ganancia permite identificar varios productos Estrella, entre ellos Bagel pavo, Pasta pomodoro, Sándwich veggie, Croissant relleno y Carrot cake.

Sin embargo, pertenecer al grupo Estrella no implica necesariamente tener un margen elevado. Algunos productos, como Quiche lorraine, generan una ganancia importante gracias a su volumen de ventas, pero presentan un margen considerablemente menor que otros productos del mismo grupo.

Por lo tanto, el volumen, la ganancia total y el margen deben analizarse conjuntamente antes de tomar decisiones comerciales.

In [92]:
catalogo_proveedores["costo_min_producto"] = (
    catalogo_proveedores
    .groupby("producto_id")["costo_unitario_proveedor"]
    .transform("min")
)

catalogo_proveedores["sobrecosto_vs_min_pct"] = (
    (
        catalogo_proveedores["costo_unitario_proveedor"]
        / catalogo_proveedores["costo_min_producto"]
    ) - 1
) * 100

comparacion_proveedores = (
    catalogo_proveedores
    .groupby("proveedor")
    .agg(
        productos_ofrecidos=("producto_id", "nunique"),
        sobrecosto_promedio_pct=("sobrecosto_vs_min_pct", "mean"),
        dias_entrega_prometidos=("dias_entrega_prometidos", "mean")
    )
    .reset_index()
)

desempeno_real = (
    df_reposiciones
    .groupby("proveedor")
    .agg(
        pedidos=("pedido_id", "count"),
        dias_entrega_real_promedio=("dias_entrega", "mean"),
        pct_pedidos_atrasados=(
            "dias_retraso",
            lambda x: (x > 0).mean() * 100
        ),
        retraso_promedio=("dias_retraso", "mean")
    )
    .reset_index()
)

resumen_proveedores = comparacion_proveedores.merge(
    desempeno_real,
    on="proveedor",
    how="left"
)

display(resumen_proveedores.round(2))

,proveedor,productos_ofrecidos,sobrecosto_promedio_pct,dias_entrega_prometidos,pedidos,dias_entrega_real_promedio,pct_pedidos_atrasados,retraso_promedio
0,Proveedor A,60,16.24,2.0,16118.0,2.16,8.10,0.16
1,Proveedor B,60,0.00,5.0,373.0,5.62,30.83,0.62
2,Proveedor C,23,7.78,3.0,NaN,NaN,NaN,NaN
3,Proveedor D,37,23.39,1.0,34176.0,1.11,5.23,0.11


In [93]:
resumen_proveedores["estado_datos"] = np.where(
    resumen_proveedores["pedidos"].isna(),
    "Sin pedidos observados",
    "Con pedidos observados"
)

display(resumen_proveedores.round(2))

,proveedor,productos_ofrecidos,sobrecosto_promedio_pct,dias_entrega_prometidos,pedidos,dias_entrega_real_promedio,pct_pedidos_atrasados,retraso_promedio,estado_datos
0,Proveedor A,60,16.24,2.0,16118.0,2.16,8.10,0.16,Con pedidos observados
1,Proveedor B,60,0.00,5.0,373.0,5.62,30.83,0.62,Con pedidos observados
2,Proveedor C,23,7.78,3.0,NaN,NaN,NaN,NaN,Sin pedidos observados
3,Proveedor D,37,23.39,1.0,34176.0,1.11,5.23,0.11,Con pedidos observados


### Hallazgo: comparación de proveedores

Los proveedores presentan distintos compromisos entre costo, velocidad y confiabilidad.

Proveedor B ofrece los menores costos, pero registra el mayor tiempo promedio de entrega y la mayor proporción de pedidos atrasados entre los proveedores con pedidos observados. Por ello, puede resultar conveniente cuando existe suficiente cobertura de inventario para esperar la reposición.

Proveedor D presenta el mayor sobrecosto, pero también la reposición más rápida y uno de los menores niveles de atraso, por lo que resulta más adecuado para necesidades urgentes.

Proveedor A ofrece una alternativa más equilibrada entre costo, rapidez y cumplimiento de los plazos.

Proveedor C presenta costos y tiempos prometidos intermedios, pero no registra pedidos efectivos durante el período analizado. Por esta razón, no existe evidencia suficiente para evaluar su desempeño real.

## Conclusiones generales

El análisis permitió identificar oportunidades de mejora en la planificación de inventario, reposición y rentabilidad de la cadena.

- La baja rotación se concentra especialmente en productos de la categoría Ensaladas y en Muffin arándanos, donde se observan niveles elevados de inventario remanente y bajo sell-through.
- El comportamiento de Muffin arándanos demuestra que un producto puede presentar exceso de inventario a nivel agregado y, al mismo tiempo, experimentar posibles quiebres en determinadas sucursales. Esto evidencia la importancia de analizar el inventario por ubicación y período.
- Maipú presenta algunos de los casos más importantes de posibles quiebres, especialmente en Muffin arándanos y Donut glaseada. En estos productos, los mayores tiempos de entrega y atrasos podrían estar contribuyendo al problema.
- Sándwich roast beef y Bagel salmón también muestran señales frecuentes de quiebre, pero con tiempos de entrega relativamente bajos, lo que apunta más hacia una necesidad de ajustar cantidades y frecuencia de reposición.
- Bagel pavo genera la mayor ganancia total del portafolio, mientras que Carrot cake presenta el mayor margen porcentual.
- La clasificación de productos muestra que un alto volumen de ventas no necesariamente implica un margen elevado, por lo que ventas, ganancia y margen deben evaluarse conjuntamente.
- Proveedor B presenta la alternativa de menor costo, pero también mayores tiempos de entrega y atrasos. Proveedor D es el más rápido, aunque con mayor costo, mientras que Proveedor A ofrece una alternativa más equilibrada.
- Proveedor C no registra pedidos efectivos durante el período, por lo que no existe evidencia suficiente para evaluar su desempeño real.

En conjunto, los resultados sugieren que la optimización del inventario no depende únicamente de reducir o aumentar las compras, sino de mejorar la planificación por producto, sucursal y momento de reposición.

In [94]:
resumen_productos_tableau = (
    inventario_producto[
        [
            "producto_id",
            "producto",
            "categoria",
            "unidades_disponibles",
            "unidades_vendidas",
            "merma_unidades",
            "stock_final_promedio",
            "tasa_merma",
            "sell_through",
            "stock_remanente_pct"
        ]
    ]
    .merge(
        baja_rotacion[
            [
                "producto_id",
                "pct_dias_venta_cero",
                "pct_dias_venta_baja",
                "pct_dias_con_stock"
            ]
        ],
        on="producto_id",
        how="left"
    )
    .merge(
        rentabilidad_producto[
            [
                "producto_id",
                "ingresos_netos",
                "costo_ventas",
                "costo_merma",
                "ganancia",
                "margen_pct",
                "clasificacion"
            ]
        ],
        on="producto_id",
        how="left"
    )
)

quiebres_tableau = quiebres_reposicion.copy()

proveedores_tableau = resumen_proveedores.copy()

In [95]:
resumen_productos_tableau.to_csv(
    "resumen_productos_tableau.csv",
    index=False
)

quiebres_tableau.to_csv(
    "quiebres_reposicion_tableau.csv",
    index=False
)

proveedores_tableau.to_csv(
    "proveedores_tableau.csv",
    index=False
)

print("Archivos exportados correctamente ✅")

Archivos exportados correctamente ✅
